In [1]:
%load_ext autoreload
import numpy as np
import jax, scipy
from jax import numpy as jnp
from qihd.utils.jax_utils import jax_device

from qihd import MIQPSolver, MIQP, QIHD, PDQP
from qihd.refiners.pdqp import PDQP


In [2]:
# Set up device; optional if device = 'cpu'
device = "cpu"
jax.config.update("jax_platforms", jax_device(device))

In [3]:
# # Generate a random MIQP instance 

seed = 42
np.random.seed(seed)

n = 30
m = 50
k = 5
nbin = int(n / 3)
U = scipy.stats.ortho_group(n, seed).rvs()
eig = np.random.normal(-10, 5, (n))
Q = U.T @ np.diag(eig) @ U
w = np.random.normal(0, 10, (n))
x = np.random.uniform(0, 5, (n))
x[:nbin] = np.random.randint(0, 2, (nbin))
lbs = np.concatenate((np.zeros(nbin), np.min(x[nbin:]) * np.ones(n - nbin)))
ubs = np.concatenate((np.ones(nbin), np.max(x[nbin:]) * np.ones(n - nbin)))
A = np.random.normal(0, 1, (m, n))
b = np.random.normal(0, np.sqrt(n), (m))
infeas = A @ x > b
A[infeas] *= -1
b[infeas] *= -1
C = np.random.uniform(0, 1, (k, n))
d = C @ x

prob = MIQP(Q, w, A=A, b=b, C=C, d=d, n_binary_vars=nbin, bounds=(lbs, ubs))

In [6]:
# Set up backend instance (QIHD)
n_shots = 100
n_steps = 10000
seed = 42
lc_pr = 10
slow_a = False
backend = QIHD(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device, lc_pr=lc_pr, slow_a=slow_a)

# Set up refiner instance (PDQP)
iterations = 10000
refiner = PDQP(iterations=iterations, device=device)

# Solve the problem using MIQPSolver
model = MIQPSolver(prob, backend, refiner)
res = model.solve()

-1075.9147228488062 1.9629086601469226e-05 0.01


In [7]:
# Validate the solution
xs, cnts = res.refined_samples, res.sample_counts

objs = np.array([prob.obj(x) for x in xs])
maxvios = np.array([prob.max_vios(x) for x in xs])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(cnts[feas][succ]) / n_shots
print(f"----- Running QiHD -----\nBackend: {backend.__class__.__name__}; Refiner: {refiner.__class__.__name__}.")
print(f"Incumbent minimum: {minima}, feasibility violations: {minimizer_vios}, success probability : {succ_prob}.")
assert minima <= -1075

----- Running QiHD -----
Backend: QIHD; Refiner: PDQP.
Incumbent minimum: -1075.9147228488062, feasibility violations: 1.9629086601469226e-05, success probability : 0.01.
